# E5, E9 — Honestidad metodológica

## Preguntas
- **E5:** ¿son confiables las confianzas del modelo? ¿ayuda abstenerse en los casos dudosos?
- **E9:** ¿el modelo es equitativo entre ejes y clases? (análogo local a SageMaker Clarify)

## Método
E5: temperature scaling en validación, medir ECE antes/después, curva risk-coverage. E9: desempeño por eje y sesgo de distribución de clases sobre las predicciones OOF.

In [ ]:
# --- Configuracion comun ---
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import common as C
R = C.RESULTS
def load(f): return json.load(open(R / f))

# Patron de dos niveles: por defecto CARGA resultados ya calculados (segundos, sin GPU).
# Para RE-EJECUTAR desde cero (requiere GPU/Bedrock), pon RECOMPUTE=True.
RECOMPUTE = False

## E5 — Calibración + abstención

In [ ]:
e5 = load("e5_calibracion.json")
print(f"ECE {e5['ece_before']:.3f} -> {e5['ece_after']:.3f} (T={e5['T']:.2f})")
import matplotlib.pyplot as plt
cov=[r['coverage']*100 for r in e5['risk_coverage']]; q=[r['qwk'] for r in e5['risk_coverage']]
plt.figure(figsize=(7,4)); plt.plot(cov,q,"-o",color="#0073bb"); plt.gca().invert_xaxis()
plt.xlabel("cobertura % (lo que SI se etiqueta)"); plt.ylabel("QWK"); plt.title("E5: abstenerse de lo dudoso sube la calidad"); plt.grid(alpha=.3); plt.show()

Al etiquetar solo el 60% más confiable, el QWK sube de 0.391 a 0.526. Para medir opinión pública es más honesto reportar % incierto que forzar una etiqueta dudosa.

## E9 — Análisis de sesgo

In [ ]:
e9 = load("e9_bias.json")
print("QWK por eje:", {k: round(v["qwk"],3) for k,v in e9["axis_performance"].items()}, "| brecha:", round(e9["axis_qwk_gap"],3))
print("Sesgo de distribucion (pp, predicho-real):", {k: round(v,1) for k,v in e9["class_bias_pp"].items()})
print("\nMatriz de confusion (fila=real, col=predicho):"); print(np.array(e9["confusion_matrix"]))

## Veredicto
El modelo es peor en **infraestructura** (eje más ambiguo) y **sub-predice la clase neutral** (sesgo anti-neutral). Los errores caen en clases vecinas (buena propiedad ordinal). Sección de ética/limitaciones obtenida a $0, sin SageMaker.